In [11]:
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def english_tokens(text):
    text = text.lower()
    return re.findall(r"[a-z]+", text)

def hindi_tokens(text):
    text = normalize_unicode(text)
    # Devanagari words; punctuation/numbers are excluded.
    return re.findall(r"[\u0900-\u097F]+", text)

def average_vectors(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

print("Core libraries loaded.")


Core libraries loaded.


In [12]:
# Q1(a): Corpus containing 50 sentences.
sentences = ['machine learning models learn patterns from data', 'machine learning algorithms improve prediction accuracy', 'deep learning models use neural networks', 'neural networks learn complex patterns from data', 'artificial intelligence systems solve complex problems', 'artificial intelligence uses machine learning methods', 'data science combines statistics and programming', 'data analysis helps researchers find useful patterns', 'data scientists build models from large datasets', 'researchers analyze data to test new ideas', 'computer vision systems analyze images', 'computer vision models detect objects in images', 'image classification assigns labels to images', 'deep learning improves image classification performance', 'natural language processing analyzes human language', 'language models learn patterns from text', 'natural language systems process words and sentences', 'text classification assigns categories to documents', 'students learn programming in computer science courses', 'students practice algorithms and data structures', 'programming skills help students solve problems', 'computer science students study machine learning', 'online courses provide lectures and programming exercises', 'online education helps students learn new skills', 'education platforms use data to track student progress', 'teachers use data analysis to understand learning outcomes', 'algorithms process data to produce useful results', 'algorithms can classify images and text', 'models learn from training data', 'training data improves machine learning models', 'large datasets help neural networks learn', 'researchers train models on large datasets', 'prediction models estimate future outcomes', 'machine learning supports prediction in many applications', 'artificial intelligence research develops new models', 'researchers develop algorithms for language processing', 'language models represent words using their contexts', 'words with similar contexts can have related meanings', 'context helps a model represent the meaning of words', 'distributional representations describe words through context', 'data models support intelligent applications', 'machine learning supports computer vision', 'neural networks support language processing', 'researchers compare models using data', 'students learn artificial intelligence and machine learning', 'software engineers build data driven applications', 'modern systems use algorithms to process information', 'prediction models help organizations make decisions', 'technology companies develop intelligent software', 'computer science research studies learning algorithms']
print("Number of sentences:", len(sentences))
assert len(sentences) >= 50


Number of sentences: 50


In [13]:
# Q1(b): Basic preprocessing.
tokenized_corpus = [english_tokens(s) for s in sentences]

print("First five processed sentences:")
for tokens in tokenized_corpus[:5]:
    print(tokens)


First five processed sentences:
['machine', 'learning', 'models', 'learn', 'patterns', 'from', 'data']
['machine', 'learning', 'algorithms', 'improve', 'prediction', 'accuracy']
['deep', 'learning', 'models', 'use', 'neural', 'networks']
['neural', 'networks', 'learn', 'complex', 'patterns', 'from', 'data']
['artificial', 'intelligence', 'systems', 'solve', 'complex', 'problems']


In [14]:
# Q1(c/d): Train Word2Vec with CBOW.
# sg=0 selects CBOW.
VECTOR_SIZE = 50
WINDOW = 3
MIN_COUNT = 1
EPOCHS = 100

cbow_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=4,
    sg=0,
    epochs=EPOCHS,
    seed=42
)

print("Architecture: CBOW")
print("Vector dimension:", cbow_model.vector_size)
print("Context window:", WINDOW)
print("Minimum word count:", MIN_COUNT)
print("Training epochs:", EPOCHS)


Architecture: CBOW
Vector dimension: 50
Context window: 3
Minimum word count: 1
Training epochs: 100


In [15]:
# Q1(e): Vocabulary size and vector dimensionality.
print("Vocabulary size:", len(cbow_model.wv))
print("Vector dimensionality:", cbow_model.wv.vector_size)


Vocabulary size: 133
Vector dimensionality: 50


In [16]:
# Q1(f): Display vectors for 10 selected words.
selected_words = [
    "machine", "learning", "data", "models", "neural",
    "networks", "language", "computer", "students", "algorithms"
]

available_words = [w for w in selected_words if w in cbow_model.wv]
sample_vectors = pd.DataFrame(
    [cbow_model.wv[w] for w in available_words],
    index=available_words
)

display(sample_vectors.round(4))


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
machine,-0.0304,-0.0071,-0.0541,0.0377,-0.0810,0.0429,-0.0443,0.1298,-0.1072,-0.0114,...,-0.0300,-0.0205,0.1073,-0.0481,0.0105,-0.0224,0.0287,0.1211,0.0512,-0.0876
learning,-0.0180,-0.0438,-0.0601,0.0192,-0.0962,0.0845,-0.0618,0.1555,-0.1337,-0.0214,...,-0.0014,-0.0446,0.1585,-0.0764,0.0288,0.0062,0.0062,0.1552,0.0645,-0.1115
data,-0.0561,-0.0272,-0.0802,0.0430,-0.1327,0.1022,-0.1060,0.1867,-0.1719,-0.0526,...,-0.0385,-0.0286,0.2098,-0.0930,-0.0060,0.0110,0.0282,0.2151,0.0654,-0.0974
models,-0.0253,-0.0496,-0.0880,0.0406,-0.1230,0.0655,-0.0831,0.1547,-0.1390,-0.0258,...,-0.0264,-0.0224,0.2033,-0.0883,0.0245,-0.0167,0.0198,0.1959,0.0601,-0.1186
neural,-0.0165,-0.0232,-0.0392,0.0397,-0.0741,0.0608,-0.0402,0.0904,-0.0739,-0.0223,...,-0.0113,-0.0185,0.1032,-0.0436,-0.0028,-0.0213,0.0273,0.0997,0.0063,-0.0512
networks,-0.0022,-0.0062,-0.0347,0.0357,-0.0704,0.0539,-0.0206,0.0761,-0.0567,-0.0139,...,-0.0030,-0.0021,0.0804,-0.0435,0.0133,-0.0177,0.0028,0.0890,0.0308,-0.0624
language,-0.0471,-0.0387,-0.0975,0.0387,-0.1176,0.0647,-0.0759,0.1513,-0.1623,-0.0291,...,-0.0096,-0.0475,0.2066,-0.0910,0.0118,-0.0094,0.0410,0.1888,0.0481,-0.1234
computer,-0.0116,-0.0152,-0.0698,0.0170,-0.0979,0.0713,-0.0895,0.1465,-0.1116,-0.0432,...,-0.0013,-0.0153,0.1697,-0.0653,0.0000,-0.0125,0.0059,0.1511,0.0670,-0.0913
students,-0.0045,-0.0149,-0.0658,0.0509,-0.0862,0.0580,-0.0606,0.1023,-0.0884,-0.0438,...,0.0017,-0.0349,0.1193,-0.0584,0.0075,-0.0023,0.0115,0.1502,0.0422,-0.0832
algorithms,-0.0369,-0.0552,-0.0910,0.0408,-0.1006,0.0971,-0.0986,0.1836,-0.1637,-0.0240,...,-0.0140,-0.0497,0.1817,-0.0768,0.0210,0.0063,0.0089,0.1748,0.0485,-0.1101


In [17]:
# Q1(g): Cosine similarity for at least five word pairs.
word_pairs = [
    ("machine", "learning"),
    ("neural", "networks"),
    ("data", "models"),
    ("language", "text"),
    ("computer", "vision")
]

similarity_results = []
for a, b in word_pairs:
    if a in cbow_model.wv and b in cbow_model.wv:
        score = cbow_model.wv.similarity(a, b)
        similarity_results.append([a, b, score])

display(pd.DataFrame(
    similarity_results,
    columns=["Word 1", "Word 2", "Cosine Similarity"]
).round(4))


,Word 1,Word 2,Cosine Similarity
0,machine,learning,0.9804
1,neural,networks,0.9506
2,data,models,0.9852
3,language,text,0.9708
4,computer,vision,0.9479


In [18]:
# Q1(h): Top similar words for at least five selected words.
for word in ["machine", "learning", "data", "models", "language"]:
    print(f"\\nTop similar words for '{word}':")
    print(cbow_model.wv.most_similar(word, topn=5))


\nTop similar words for 'machine':
[('the', 0.9827932715415955), ('words', 0.9820020794868469), ('to', 0.9818856120109558), ('prediction', 0.9809277057647705), ('science', 0.9805852770805359)]
\nTop similar words for 'learning':
[('model', 0.9867932796478271), ('helps', 0.9864066243171692), ('models', 0.9856755137443542), ('analysis', 0.9855162501335144), ('algorithms', 0.9851818680763245)]
\nTop similar words for 'data':
[('can', 0.9900137186050415), ('track', 0.9894235134124756), ('to', 0.9890443682670593), ('use', 0.9870620965957642), ('helps', 0.9864781498908997)]
\nTop similar words for 'models':
[('researchers', 0.9899434447288513), ('to', 0.9895901679992676), ('language', 0.9886512756347656), ('learning', 0.9856756329536438), ('the', 0.9856593608856201)]
\nTop similar words for 'language':
[('to', 0.9903523921966553), ('models', 0.9886512756347656), ('contexts', 0.9884014129638672), ('the', 0.9865098595619202), ('represent', 0.986194372177124)]


### Q1(i): Distributional hypothesis and CBOW

The **distributional hypothesis** says that words appearing in similar contexts tend to have similar meanings or related usage.

**CBOW (Continuous Bag of Words)** predicts a target word from the surrounding context words. During training, the model adjusts the word vectors so that useful contextual relationships are represented in the embedding space.

The resulting Word2Vec vectors are **dense static word embeddings**: each word has one vector learned from the training corpus.


### Observation

The quality of CBOW embeddings depends on the corpus, vocabulary, context window, vector dimension, minimum word count and number of training epochs. A small educational corpus produces useful demonstrations, but substantially larger corpora generally produce richer embeddings.
